# Data Wrangling with Polars

This notebook covers data wrangling techniques using **Polars**, a fast DataFrame library built on Apache Arrow.

We work with two datasets from the `data/` folder:
- `ingatlan-listings.parquet` — 9M+ price observations per property per day  
- `ingatlan-details.parquet` — property-level attributes

In [1]:
import polars as pl

In [2]:
listings_df = pl.read_parquet("data/ingatlan-listings.parquet")
listings_df

id,day,price
i64,datetime[μs],i64
576781,2025-06-21 00:00:00,362466
576781,2025-05-22 00:00:00,362340
576781,2025-07-31 00:00:00,358893
576781,2025-06-04 00:00:00,362934
576781,2025-07-01 00:00:00,359370
…,…,…
35237096,2026-02-19 00:00:00,283320
35237099,2026-02-19 00:00:00,180000
35237103,2026-02-19 00:00:00,260654


In [3]:
details_df = pl.read_parquet("data/ingatlan-details.parquet")
details_df

loc,city,price,m2,rooms,balcony,id
str,str,i64,i64,str,str,i64
"""IX. kerület, Lónyay utca ""","""Budapest""",498862,95,"""3""","""""",7065343
"""Gárdony, Agárd ""","""Gárdony""",120000,25,"""1 fél""","""""",33544237
"""VI. kerület, Hajós utca ""","""Budapest""",306992,41,"""1""","""""",32889588
"""II. kerület, Török utca ""","""Budapest""",435000,125,"""3""","""10 m2""",33717225
"""V. kerület, Vörösmarty tér ""","""Budapest""",1290000,205,"""6""","""""",33611258
…,…,…,…,…,…,…
"""Budapest VI. kerület, Külső-Te…","""Budapest""",400000,140,"""3""",null,35234253
"""Budapest XII. kerület, Kiss Já…","""Budapest""",195000,56,"""2""",null,35234011
"""Budapest VIII. kerület, Corvin…","""Budapest""",490000000,70,"""2 + 1 fél""","""8 m2""",35232596


In [4]:
listings_ldf = pl.scan_parquet("data/ingatlan-listings.parquet")
listings_ldf  # shows the query plan, not the data

In [5]:
details_ldf = pl.scan_parquet("data/ingatlan-details.parquet")
details_ldf

In [6]:
print(listings_ldf.explain(optimized=True))

Parquet SCAN [data/ingatlan-listings.parquet]
PROJECT */3 COLUMNS
ESTIMATED ROWS: 9130075


In [7]:
listings_ldf.collect()

id,day,price
i64,datetime[μs],i64
576781,2025-06-21 00:00:00,362466
576781,2025-05-22 00:00:00,362340
576781,2025-07-31 00:00:00,358893
576781,2025-06-04 00:00:00,362934
576781,2025-07-01 00:00:00,359370
…,…,…
35237096,2026-02-19 00:00:00,283320
35237099,2026-02-19 00:00:00,180000
35237103,2026-02-19 00:00:00,260654


## Q1 — Filter: properties in Budapest

In [8]:
details_df.filter(pl.col("city") == "Budapest")

loc,city,price,m2,rooms,balcony,id
str,str,i64,i64,str,str,i64
"""IX. kerület, Lónyay utca ""","""Budapest""",498862,95,"""3""","""""",7065343
"""VI. kerület, Hajós utca ""","""Budapest""",306992,41,"""1""","""""",32889588
"""II. kerület, Török utca ""","""Budapest""",435000,125,"""3""","""10 m2""",33717225
"""V. kerület, Vörösmarty tér ""","""Budapest""",1290000,205,"""6""","""""",33611258
"""XII. kerület, Kissvábhegy ""","""Budapest""",270000,46,"""1 + 1 fél""","""23 m2""",33339351
…,…,…,…,…,…,…
"""Budapest XVI. kerület, Szérű u…","""Budapest""",190000,43,"""1 + 1 fél""",null,35233670
"""Budapest XIV. kerület, Dorozsm…","""Budapest""",300000,51,"""2""","""6 m2""",35232659
"""Budapest VI. kerület, Külső-Te…","""Budapest""",400000,140,"""3""",null,35234253


## Q2 — Filter: expensive listings

In [9]:
listings_df.filter(pl.col("price") > 500_000)

id,day,price
i64,datetime[μs],i64
1000603,2025-05-26 00:00:00,1331154
1000603,2025-08-23 00:00:00,1315941
1000603,2024-12-27 00:00:00,1368411
1000603,2025-03-19 00:00:00,1313235
1000603,2025-02-20 00:00:00,1324851
…,…,…
35236264,2026-02-19 00:00:00,642192
35236653,2026-02-18 00:00:00,600000
35236653,2026-02-19 00:00:00,600000


## Q3 — Sort: 10 largest properties

In [10]:
details_df.sort("m2", descending=True).head(10)

loc,city,price,m2,rooms,balcony,id
str,str,i64,i64,str,str,i64
"""Békéscsaba, Körösi utca 10.""","""Békéscsaba""",80000,70000,"""1 + 2 fél""",null,34329808
"""V. kerület, Dorottya utca ""","""Budapest""",646051,63333,"""2""","""4 m2""",33882767
"""Budapest IV. kerület, Árpád út""","""Budapest""",250000,57000,"""2""","""3 m2""",7004349
"""Békéscsaba, Penza-lakótelep""","""Békéscsaba""",90000,52000,"""2""",null,34307899
"""Budapest VIII. kerület, József…","""Budapest""",295000,42090,"""1 + 1 fél""","""3 m2""",34840301
"""Debrecen, Belváros""","""Debrecen""",250000,12499,"""3""","""3 m2""",34736358
"""Paks, Liget utca 11.""","""Paks""",30508000,7325,"""349""",null,34494256
"""Budapest XIII. kerület, Béke t…","""Budapest""",270000,5656,"""2""","""4 m2""",34792724
"""Miskolc, Thököly Imre utca""","""Miskolc""",100000,4711,"""1 + 1 fél""","""4 m2""",34865266


## Q4 — Select: keep a subset of columns

In [11]:
details_df.select("id", "city", "m2", "price")

id,city,m2,price
i64,str,i64,i64
7065343,"""Budapest""",95,498862
33544237,"""Gárdony""",25,120000
32889588,"""Budapest""",41,306992
33717225,"""Budapest""",125,435000
33611258,"""Budapest""",205,1290000
…,…,…,…
35234253,"""Budapest""",140,400000
35234011,"""Budapest""",56,195000
35232596,"""Budapest""",70,490000000


## Q5 — Filter + sort: cheap and small properties

In [12]:
details_df.filter((pl.col("m2") < 40) & (pl.col("price") < 100_000)).sort("price")

loc,city,price,m2,rooms,balcony,id
str,str,i64,i64,str,str,i64
"""Szolnok, Glykais Gyula utca""","""Szolnok""",80,38,"""1""",null,34266762
"""Budapest XV. kerület, Páskom p…","""Budapest""",90,10,"""1 + 2 fél""",null,34297777
"""Budapest XIII. kerület, Pannón…","""Budapest""",140,20,"""1""",null,34215860
"""Budapest IX. kerület, Haller u…","""Budapest""",150,36,"""1""",null,34281967
"""Budapest VIII. kerület, Tömő u…","""Budapest""",170,33,"""1""","""5 m2""",34235080
…,…,…,…,…,…,…
"""Budapest XIX. kerület, Hamvas …","""Budapest""",99990,12,"""3""",null,34243771
"""Budapest VIII. kerület, Üllői …","""Budapest""",99990,10,"""4""",null,34292095
"""Budapest XX. kerület, Nagyszől…","""Budapest""",99990,13,"""1""",null,32685015


## Q6 — Group by: number of properties per city

In [13]:
city_counts = details_df.group_by("city").len().sort("len", descending=True)

In [14]:
import plotly.express as px

px.bar(
    city_counts.head(20),
    x="len",
    y="city",
    orientation="h",
    title="Top 20 Cities by Property Count",
    labels={"len": "Number of Properties", "city": "City"},
)


## Q7 — Group by + agg: average price and area per city

In [16]:
(
    details_df.group_by("city")
    .agg(
        avg_price=pl.col("price").mean(),
        avg_m2=pl.col("m2").mean(),
    )
    .sort("avg_price", descending=True)
)

city,avg_price,avg_m2
str,f64,f64
"""Pornóapáti""",1.0000e10,50.0
"""Kiskunlacháza""",5.51025e7,59.0
"""Verpelét""",3.5045e7,51.5
"""Aggtelek""",3.25325e7,18.5
"""Pilisszántó""",3.0107e7,48.333333
…,…,…
"""Kétegyháza""",47500.0,59.0
"""Darány""",35000.0,69.0
"""Újszász""",35000.0,20.0


## Q8 — String filter: Budapest district properties

In [17]:
details_df.filter(
    pl.col("loc").str.contains("kerület"),
)

loc,city,price,m2,rooms,balcony,id
str,str,i64,i64,str,str,i64
"""IX. kerület, Lónyay utca ""","""Budapest""",498862,95,"""3""","""""",7065343
"""VI. kerület, Hajós utca ""","""Budapest""",306992,41,"""1""","""""",32889588
"""II. kerület, Török utca ""","""Budapest""",435000,125,"""3""","""10 m2""",33717225
"""V. kerület, Vörösmarty tér ""","""Budapest""",1290000,205,"""6""","""""",33611258
"""XII. kerület, Kissvábhegy ""","""Budapest""",270000,46,"""1 + 1 fél""","""23 m2""",33339351
…,…,…,…,…,…,…
"""Budapest XVI. kerület, Szérű u…","""Budapest""",190000,43,"""1 + 1 fél""",null,35233670
"""Budapest XIV. kerület, Dorozsm…","""Budapest""",300000,51,"""2""","""6 m2""",35232659
"""Budapest VI. kerület, Külső-Te…","""Budapest""",400000,140,"""3""",null,35234253


## Q9 — Derived column: has balcony flag

In [19]:
details_df.with_columns(
    has_balcony=pl.col("balcony").is_not_null()
    & (pl.col("balcony").str.len_chars() > 0)
)

loc,city,price,m2,rooms,balcony,id,has_balcony
str,str,i64,i64,str,str,i64,bool
"""IX. kerület, Lónyay utca ""","""Budapest""",498862,95,"""3""","""""",7065343,false
"""Gárdony, Agárd ""","""Gárdony""",120000,25,"""1 fél""","""""",33544237,false
"""VI. kerület, Hajós utca ""","""Budapest""",306992,41,"""1""","""""",32889588,false
"""II. kerület, Török utca ""","""Budapest""",435000,125,"""3""","""10 m2""",33717225,true
"""V. kerület, Vörösmarty tér ""","""Budapest""",1290000,205,"""6""","""""",33611258,false
…,…,…,…,…,…,…,…
"""Budapest VI. kerület, Külső-Te…","""Budapest""",400000,140,"""3""",null,35234253,false
"""Budapest XII. kerület, Kiss Já…","""Budapest""",195000,56,"""2""",null,35234011,false
"""Budapest VIII. kerület, Corvin…","""Budapest""",490000000,70,"""2 + 1 fél""","""8 m2""",35232596,true


## Q10 — Time filter: listings recorded in 2026

In [20]:
listings_df.filter(pl.col("day").dt.year() == 2026)

id,day,price
i64,datetime[μs],i64
990996,2026-01-16 00:00:00,424545
990996,2026-01-06 00:00:00,423698
990996,2026-02-13 00:00:00,416273
990996,2026-01-07 00:00:00,423698
990996,2026-01-28 00:00:00,420288
…,…,…
35237096,2026-02-19 00:00:00,283320
35237099,2026-02-19 00:00:00,180000
35237103,2026-02-19 00:00:00,260654


## Q11 — Join: attach city to every listing

In [21]:
listings_with_city = listings_df.join(
    details_df.select("id", "city"),
    on="id",
    how="left",
)
listings_with_city

id,day,price,city
i64,datetime[μs],i64,str
576781,2025-06-21 00:00:00,362466,"""Budapest"""
576781,2025-05-22 00:00:00,362340,"""Budapest"""
576781,2025-07-31 00:00:00,358893,"""Budapest"""
576781,2025-06-04 00:00:00,362934,"""Budapest"""
576781,2025-07-01 00:00:00,359370,"""Budapest"""
…,…,…,…
35237096,2026-02-19 00:00:00,283320,"""Budapest"""
35237099,2026-02-19 00:00:00,180000,"""Budapest"""
35237103,2026-02-19 00:00:00,260654,"""Budapest"""


## Q12 — Time series: average listing price per calendar month

In [25]:
monthly_avg = (
    listings_df.with_columns(
        month=pl.col("day").dt.truncate("1mo"),
    )
    .group_by("month")
    .agg(
        avg_price=pl.col("price").mean(),
    )
    .sort("month")
)
monthly_avg

month,avg_price
datetime[μs],f64
2023-10-01 00:00:00,383056.415029
2023-11-01 00:00:00,479154.4786
2023-12-01 00:00:00,562262.86758
2024-01-01 00:00:00,472554.305483
2024-02-01 00:00:00,429505.018368
…,…
2025-10-01 00:00:00,419891.193274
2025-11-01 00:00:00,388305.274227
2025-12-01 00:00:00,421971.556197


In [26]:
px.line(
    monthly_avg,
    x="month",
    y="avg_price",
    title="Average Listing Price per Month",
    labels={"month": "Month", "avg_price": "Avg Price (000 HUF)"},
)


## Q13 — Derived column: price per square metre

In [27]:
(
    details_df.with_columns(
        price_per_m2=pl.col("price") / pl.col("m2"),
    )
    .sort("price_per_m2", descending=True)
    .head(10)
)

loc,city,price,m2,rooms,balcony,id,price_per_m2
str,str,i64,i64,str,str,i64,f64
"""IX. kerület, Dandár köz ""","""Budapest""",42000000000,40,"""1 + 1 fél""","""10 m2""",34115307,1.0500e9
"""Budapest III., Nyár utca 21.""","""Budapest""",10000000000,30,"""1 + 1 fél""","""5 m2""",34180278,3.3333e8
"""Pornóapáti, Fő utca""","""Pornóapáti""",10000000000,50,"""5 + 5 fél""","""50 m2""",35002260,2e8
"""VII. kerület, Wesselényi utca …","""Budapest""",3800000000,38,"""1""","""""",31401100,1e8
"""XXI. kerület, Puli sétány 14.""","""Budapest""",800000000,12,"""1""","""""",33966162,6.6667e7
"""Budapest III. kerület, Körtvél…","""Budapest""",800000000,12,"""4""","""30 m2""",35162308,6.6667e7
"""Budapest II. kerület, Vérhalom""","""Budapest""",2750000000,109,"""5""","""91 m2""",34653222,2.5229e7
"""Siófok, Beszédes József sétány…","""Siófok""",1125000000,55,"""2 + 1 fél""","""10 m2""",34117725,2.0455e7
"""Budapest VI. kerület, Dalszính…","""Budapest""",1590000000,115,"""4""","""3 m2""",35110092,1.3826e7


## Q14 — String extract: numeric balcony area

In [28]:
details_df.with_columns(
    balcony_m2=pl.col("balcony").str.extract(r"(\d+\.?\d*)").cast(pl.Float64),
)

loc,city,price,m2,rooms,balcony,id,balcony_m2
str,str,i64,i64,str,str,i64,f64
"""IX. kerület, Lónyay utca ""","""Budapest""",498862,95,"""3""","""""",7065343,null
"""Gárdony, Agárd ""","""Gárdony""",120000,25,"""1 fél""","""""",33544237,null
"""VI. kerület, Hajós utca ""","""Budapest""",306992,41,"""1""","""""",32889588,null
"""II. kerület, Török utca ""","""Budapest""",435000,125,"""3""","""10 m2""",33717225,10.0
"""V. kerület, Vörösmarty tér ""","""Budapest""",1290000,205,"""6""","""""",33611258,null
…,…,…,…,…,…,…,…
"""Budapest VI. kerület, Külső-Te…","""Budapest""",400000,140,"""3""",null,35234253,null
"""Budapest XII. kerület, Kiss Já…","""Budapest""",195000,56,"""2""",null,35234011,null
"""Budapest VIII. kerület, Corvin…","""Budapest""",490000000,70,"""2 + 1 fél""","""8 m2""",35232596,8.0


## Q15 — Window function: price rank within each city

In [30]:
(
    details_df.with_columns(
        city_price_rank=pl.col("price")
        .rank(method="dense", descending=True)
        .over("city"),
    ).sort(["city", "city_price_rank"])
)

loc,city,price,m2,rooms,balcony,id,city_price_rank
str,str,i64,i64,str,str,i64,u32
"""Abony, Erzsébet királyné utca ""","""Abony""",100000,50,"""1""","""""",33735519,1
"""Abádszalók, Széchenyi István ú…","""Abádszalók""",98000,33,"""1""",null,34792805,1
"""Acsa, Kossuth út 37.""","""Acsa""",120000,48,"""2""",null,34626724,1
"""Adony, Rákóczi utca ""","""Adony""",599000,70,"""2 + 2 fél""","""5 m2""",32741096,1
"""Adony, Petőfi utca 2.""","""Adony""",160000,56,"""1 + 1 fél""","""6 m2""",34365197,2
…,…,…,…,…,…,…,…
"""Őrbottyán, Pest megye""","""Őrbottyán""",290000,63,"""2""",null,34861368,2
"""Őrbottyán, Csomádi út""","""Őrbottyán""",190000,63,"""2""","""12 m2""",34386225,3
"""Őrbottyán, Kvassay telep""","""Őrbottyán""",190000,63,"""2""","""12 m2""",34419217,3


## Q16 — Time series: most recent price per property

In [31]:
listings_df.sort("day", descending=True).group_by("id").first()

id,day,price
i64,datetime[μs],i64
35092460,2025-11-17 00:00:00,295000
34706949,2025-04-17 00:00:00,210000
34989202,2025-09-29 00:00:00,220000
34568705,2025-03-11 00:00:00,299355
34385173,2024-11-21 00:00:00,155000
…,…,…
34535999,2025-06-17 00:00:00,230000
34397591,2024-10-14 00:00:00,200000
33877164,2023-12-03 00:00:00,200000


## Q17 — Join + string: top 10 cities by average price-per-m²

In [32]:
(
    details_df.with_columns(
        price_per_m2=pl.col("price") / pl.col("m2"),
    )
    .group_by("city")
    .agg(
        avg_price_per_m2=pl.col("price_per_m2").mean(),
    )
    .sort("avg_price_per_m2", descending=True)
    .head(10)
)

city,avg_price_per_m2
str,f64
"""Pornóapáti""",2e8
"""Aggtelek""",2.7096e6
"""Pilisszántó""",752074.074074
"""Verpelét""",700849.056604
"""Kiskunlacháza""",552298.369565
"""Répcelak""",314167.175658
"""Telki""",242494.279405
"""Kisláng""",240000.0
"""Remeteszőlős""",205557.13155


## Q18 — String parse + join + time series: monthly average price by room count

In [35]:
room_counts = details_df.with_columns(
    room_count=pl.col("rooms").str.extract(r"^(\d+)").cast(pl.Int64),
).select("id", "room_count")

monthly_room = (
    listings_df.join(room_counts, on="id", how="left")
    .filter(pl.col("room_count").is_between(1, 5))
    .with_columns(
        month=pl.col("day").dt.truncate("1mo"),
    )
    .group_by("month", "room_count")
    .agg(
        avg_price=pl.col("price").mean(),
    )
    .sort("month", "room_count")
)

In [36]:
px.line(
    monthly_room,
    x="month",
    y="avg_price",
    color="room_count",
    title="Monthly Average Listing Price by Room Count",
    labels={
        "month": "Month",
        "avg_price": "Avg Price (000 HUF)",
        "room_count": "Rooms",
    },
)


## Q19 — Window function: properties with the most price drops

In [37]:
(
    listings_df.sort(["id", "day"])
    .with_columns(
        prev_price=pl.col("price").shift(1).over("id"),
    )
    .with_columns(
        is_drop=(pl.col("price") < pl.col("prev_price")),
    )
    .group_by("id")
    .agg(
        drop_count=pl.col("is_drop").sum(),
    )
    .sort("drop_count", descending=True)
    .head(10)
)

id,drop_count
i64,u32
33669284,228
33669286,227
33668824,225
33669352,223
33668816,219
33778887,219
33778889,213
33669282,211
33280803,206


## Q20 — Multi-step: cities with the biggest average price drop 2025 → 2026

In [39]:
city_lookup = details_df.select("id", "city")

avg_by_city_year = (
    listings_df.join(city_lookup, on="id", how="left")
    .with_columns(year=pl.col("day").dt.year())
    .filter(pl.col("year").is_in([2025, 2026]))
    .group_by("city", "year")
    .agg(avg_price=pl.col("price").mean())
)

avg_2025 = avg_by_city_year.filter(pl.col("year") == 2025).select(
    "city", avg_price_2025=pl.col("avg_price")
)
avg_2026 = avg_by_city_year.filter(pl.col("year") == 2026).select(
    "city", avg_price_2026=pl.col("avg_price")
)

(
    avg_2025.join(avg_2026, on="city", how="inner")
    .with_columns(
        price_drop=pl.col("avg_price_2025") - pl.col("avg_price_2026"),
    )
    .sort("price_drop", descending=True)
    .head(10)
)

city,avg_price_2025,avg_price_2026,price_drop
str,f64,f64,f64
"""Aggtelek""",1.1295e6,65000.0,1.0645e6
"""Répcelak""",897119.565217,100000.0,797119.565217
"""Remeteszőlős""",1.0900e6,354761.904762,735216.237315
"""Kiskunlacháza""",685478.4689,140000.0,545478.4689
"""Dunaszentmiklós""",661219.512195,200000.0,461219.512195
"""Paks""",645387.933108,221684.435028,423703.49808
"""Levél""",564573.039384,188192.517483,376380.521901
"""Szántód""",775022.38806,404622.641509,370399.74655
"""Adony""",547685.714286,266033.707865,281652.006421


## Q21 — Window + rolling: first day a property's rolling price crossed below the global median

In [40]:
global_median = listings_df["price"].median()

(
    listings_df.sort(["id", "day"])
    .with_columns(
        rolling_avg=pl.col("price").rolling_mean(window_size=7).over("id"),
    )
    .filter(
        pl.col("rolling_avg").is_not_null() & (pl.col("rolling_avg") < global_median)
    )
    .group_by("id")
    .agg(
        first_cross_day=pl.col("day").min(),
    )
    .sort("first_cross_day")
)

id,first_cross_day
i64,datetime[μs]
21420566,2023-11-23 00:00:00
22717849,2023-11-23 00:00:00
28536473,2023-11-23 00:00:00
29001032,2023-11-23 00:00:00
29805065,2023-11-23 00:00:00
…,…
35227089,2026-02-19 00:00:00
35227125,2026-02-19 00:00:00
35227128,2026-02-19 00:00:00


## Q22 — Z-score anomaly detection per property

In [41]:
(
    listings_df.with_columns(
        mean_price=pl.col("price").mean().over("id"),
        std_price=pl.col("price").std().over("id"),
    )
    .filter(pl.col("std_price") > 0)
    .with_columns(
        z_score=(pl.col("price") - pl.col("mean_price")) / pl.col("std_price"),
    )
    .sort(pl.col("z_score").abs(), descending=True)
    .head(20)
    .select("id", "day", "price", "z_score")
)

id,day,price,z_score
i64,datetime[μs],i64,f64
33892855,2023-12-03 00:00:00,80000000,23.3024
34086344,2024-06-20 00:00:00,7000000000,22.559999
33741248,2023-10-20 00:00:00,230000,22.494488
33594706,2023-10-20 00:00:00,100000,22.383074
33594704,2023-10-20 00:00:00,150000,22.181119
…,…,…,…
33969087,2024-01-30 00:00:00,140000000,16.643425
33769305,2023-10-20 00:00:00,280000,16.553055
31717864,2024-08-10 00:00:00,210000,-16.522822


## Q23 — String regex: Budapest district median price per m²

In [44]:
districts = (
    details_df.filter(pl.col("city") == "Budapest")
    .with_columns(
        district=pl.col("loc").str.extract(r"^([IVX]+)\. kerület"),
        price_per_m2=pl.col("price") / pl.col("m2"),
    )
    .filter(pl.col("district").is_not_null())
    .group_by("district")
    .agg(
        median_price_per_m2=pl.col("price_per_m2").median(),
        listing_count=pl.len(),
    )
    .sort("median_price_per_m2", descending=True)
)

In [45]:
px.bar(
    districts,
    x="district",
    y="median_price_per_m2",
    title="Budapest Districts — Median Price per m²",
    labels={"district": "District", "median_price_per_m2": "Median Price/m² (000 HUF)"},
)


## Q24 — Cohort analysis: average price by months since first listing

In [46]:
first_listing_month = listings_df.group_by("id").agg(
    cohort_month=pl.col("day").min().dt.truncate("1mo"),
)
(
    listings_df.join(first_listing_month, on="id")
    .with_columns(
        month=pl.col("day").dt.truncate("1mo"),
    )
    .with_columns(
        months_since_first=(
            (pl.col("month").dt.year() - pl.col("cohort_month").dt.year()) * 12
            + pl.col("month").dt.month()
            - pl.col("cohort_month").dt.month()
        ).cast(pl.Int32),
    )
    .filter(pl.col("months_since_first").is_between(0, 5))
    .group_by("cohort_month", "months_since_first")
    .agg(
        avg_price=pl.col("price").mean(),
        property_count=pl.col("id").n_unique(),
    )
    .sort("cohort_month", "months_since_first")
)

cohort_month,months_since_first,avg_price,property_count
datetime[μs],i32,f64,u32
2023-10-01 00:00:00,0,383056.415029,11298
2023-10-01 00:00:00,1,408362.91333,9137
2023-10-01 00:00:00,2,422607.261309,6081
2023-10-01 00:00:00,3,444596.934844,4815
2023-10-01 00:00:00,4,460668.023925,3595
…,…,…,…
2025-12-01 00:00:00,1,333592.803022,3560
2025-12-01 00:00:00,2,336653.49633,1693
2026-01-01 00:00:00,0,429284.255808,7069


## Q25 — Price spike detection with neighbour comparison

In [ ]:
(
    listings_df.sort(["id", "day"])
    .with_columns(
        prev_price=pl.col("price").shift(1).over("id"),
        next_price=pl.col("price").shift(-1).over("id"),
    )
    .with_columns(
        is_spike=(pl.col("price") > pl.col("prev_price"))
        & (pl.col("price") > pl.col("next_price")),
    )
    .group_by("id")
    .agg(
        spike_count=pl.col("is_spike").sum(),
    )
    .sort("spike_count", descending=True)
    .head(10)
    .join(
        details_df.select("id", "city", "loc"),
        on="id",
        how="left",
    )
)

id,spike_count,city,loc
i64,u32,str,str
33778887,76,"""Budapest""","""VI. kerület, Csengery utca """
33669284,72,"""Budapest""","""VI. kerület, Szondi utca """
33668816,70,"""Budapest""","""VI. kerület, Aradi utca """
33668824,68,"""Budapest""","""VI. kerület, Aradi utca """
33669282,68,"""Budapest""","""VI. kerület, Szív utca """
33280803,67,"""Budapest""","""II. kerület, Szemlőhegy """
33669286,67,"""Budapest""","""VI. kerület, Szív utca """
33778886,67,"""Budapest""","""VI. kerület, Szív utca """
33670566,65,"""Budapest""","""VI. kerület, Podmaniczky utca """


## Q26 — Pairwise monthly price correlation between cities

In [ ]:
top10_cities = (
    details_df.group_by("city")
    .len()
    .sort("len", descending=True)
    .head(10)
    .get_column("city")
    .to_list()
)

monthly_city = (
    listings_df.join(details_df.select("id", "city"), on="id", how="left")
    .filter(pl.col("city").is_in(top10_cities))
    .with_columns(month=pl.col("day").dt.truncate("1mo"))
    .group_by("city", "month")
    .agg(avg_price=pl.col("price").mean())
)

(
    monthly_city.join(
        monthly_city.rename({"city": "city_b", "avg_price": "avg_price_b"}),
        on="month",
    )
    .filter(pl.col("city") < pl.col("city_b"))
    .group_by("city", "city_b")
    .agg(
        correlation=pl.corr("avg_price", "avg_price_b"),
    )
    .sort("correlation", descending=True)
)

city,city_b,correlation
str,str,f64
"""Debrecen""","""Pécs""",0.842077
"""Pécs""","""Székesfehérvár""",0.480446
"""Kecskemét""","""Nyíregyháza""",0.479143
"""Győr""","""Székesfehérvár""",0.358001
"""Budapest""","""Győr""",0.33845
…,…,…
"""Győr""","""Nyíregyháza""",-0.197304
"""Budapest""","""Miskolc""",-0.210701
"""Miskolc""","""Pécs""",-0.21261


## Q27 — First day a property crossed its city median from below to above (2025)

In [47]:
city_median = details_df.group_by("city").agg(
    city_median_price=pl.col("price").median(),
)
(
    listings_df.join(details_df.select("id", "city"), on="id", how="left")
    .join(city_median, on="city", how="left")
    .sort(["id", "day"])
    .with_columns(
        above_median=(pl.col("price") > pl.col("city_median_price")),
    )
    .with_columns(
        prev_above_median=pl.col("above_median").shift(1).over("id"),
    )
    .filter(
        pl.col("above_median")
        & (pl.col("prev_above_median") == False)
        & (pl.col("day").dt.year() == 2025)
    )
    .group_by("id")
    .agg(
        first_cross_up_day=pl.col("day").min(),
    )
    .sort("first_cross_up_day")
)

id,first_cross_up_day
i64,datetime[μs]
34301851,2025-01-01 00:00:00
31314325,2025-01-03 00:00:00
32372731,2025-01-03 00:00:00
32822725,2025-01-03 00:00:00
33865934,2025-01-03 00:00:00
…,…
32860098,2025-12-30 00:00:00
35143466,2025-12-30 00:00:00
31579230,2025-12-31 00:00:00


## Q28 — Price acceleration: top 10 most erratic properties

In [48]:
(
    listings_df.sort(["id", "day"])
    .with_columns(
        velocity=(pl.col("price") - pl.col("price").shift(1).over("id")).cast(
            pl.Float64
        ),
    )
    .with_columns(
        acceleration=pl.col("velocity") - pl.col("velocity").shift(1).over("id"),
    )
    .group_by("id")
    .agg(
        mean_abs_acceleration=pl.col("acceleration").abs().mean(),
        obs_count=pl.len(),
    )
    .filter(pl.col("obs_count") >= 60)
    .sort("mean_abs_acceleration", descending=True)
    .head(10)
    .join(
        details_df.select("id", "city", "loc"),
        on="id",
        how="left",
    )
)

id,mean_abs_acceleration,obs_count,city,loc
i64,f64,u32,str,str
31563753,1.8670e8,63,"""Budapest""","""I. kerület, Országház utca 34."""
34086344,5.5056e7,511,"""Budapest""","""XXIII. kerület, Tartsay utca """
34369714,4.3545e7,80,"""Budapest""","""Budapest II. kerület, Szépvölg…"
34716728,3.5383e7,159,"""Budapest""","""Budapest XIII. kerület, Katona…"
33792844,2.8772e7,152,"""Budapest""","""XII. kerület, Németvölgyi út """
34864338,2.4514e7,93,"""Budapest""","""Budapest VI. kerület, Jókai ut…"
35110092,2.4451e7,67,"""Budapest""","""Budapest VI. kerület, Dalszính…"
34278570,2.2930e7,63,"""Budapest""","""Budapest I. kerület, Víziváros…"
34653222,2.1634e7,256,"""Budapest""","""Budapest II. kerület, Vérhalom"""


## Q29 — Cohort retention rate

In [49]:
(
    listings_df.group_by("id")
    .agg(
        first_day=pl.col("day").min(),
        last_day=pl.col("day").max(),
    )
    .with_columns(
        cohort_month=pl.col("first_day").dt.truncate("1mo"),
        survived_90d=((pl.col("last_day") - pl.col("first_day")).dt.total_days() >= 90),
    )
    .group_by("cohort_month")
    .agg(
        total=pl.len(),
        survived=pl.col("survived_90d").sum(),
    )
    .with_columns(
        retention_rate=pl.col("survived").cast(pl.Float64) / pl.col("total"),
    )
    .sort("cohort_month")
)

cohort_month,total,survived,retention_rate
datetime[μs],u32,u32,f64
2023-10-01 00:00:00,11298,5329,0.471676
2023-11-01 00:00:00,8710,2704,0.310448
2023-12-01 00:00:00,5375,1601,0.29786
2024-01-01 00:00:00,7978,2298,0.288042
2024-02-01 00:00:00,7446,2015,0.270615
…,…,…,…
2025-10-01 00:00:00,6861,1527,0.222562
2025-11-01 00:00:00,6660,844,0.126727
2025-12-01 00:00:00,4974,0,0.0


## Q30 — Listing gap fraction: missing days on the market

In [50]:
(
    listings_df.group_by("id")
    .agg(
        first_day=pl.col("day").min(),
        last_day=pl.col("day").max(),
        observed_days=pl.len(),
    )
    .with_columns(
        expected_days=(
            (pl.col("last_day") - pl.col("first_day")).dt.total_days() + 1
        ).cast(pl.Int64),
    )
    .with_columns(
        gap_fraction=(
            (pl.col("expected_days") - pl.col("observed_days")).cast(pl.Float64)
            / pl.col("expected_days")
        ),
    )
    .filter(pl.col("observed_days") >= 30)
    .sort("gap_fraction", descending=True)
    .head(10)
    .join(
        details_df.select("id", "city", "loc"),
        on="id",
        how="left",
    )
)

id,first_day,last_day,observed_days,expected_days,gap_fraction,city,loc
i64,datetime[μs],datetime[μs],u32,i64,f64,str,str
32511117,2023-10-20 00:00:00,2026-02-18 00:00:00,30,853,0.96483,"""Budapest""","""XIII. kerület, Váci út """
33847846,2023-11-05 00:00:00,2026-02-17 00:00:00,32,836,0.961722,"""Debrecen""","""Debrecen, Antall József utca """
33892798,2023-12-03 00:00:00,2026-02-18 00:00:00,31,809,0.961681,"""Budapest""","""X. kerület, Gém utca """
33816150,2023-10-20 00:00:00,2026-01-28 00:00:00,32,832,0.961538,"""Budapest""","""XI. kerület, Bukarest utca """
33869229,2023-11-20 00:00:00,2026-01-04 00:00:00,30,777,0.96139,"""Vác""","""Vác, Deákvár """
33451449,2023-10-20 00:00:00,2026-02-19 00:00:00,33,854,0.961358,"""Budapest""","""XIV. kerület, Beckó utca """
33926875,2024-01-07 00:00:00,2026-02-19 00:00:00,30,775,0.96129,"""Százhalombatta""","""Százhalombatta, Erkel Ferenc k…"
32355486,2023-12-10 00:00:00,2026-02-11 00:00:00,31,795,0.961006,"""Budapest""","""XI. kerület, Galambóc utca 16."""
33858952,2023-11-20 00:00:00,2025-12-17 00:00:00,30,759,0.960474,"""Kecskemét""","""Kecskemét, Széchenyiváros """
